# Initialization

In [1]:
import os
temp_path = os.getcwd().split('\\')
project_dir = '\\'.join(temp_path[:temp_path.index('SOURCE') + 1])

In [5]:
import sys

sys.path.insert(0, f'{project_dir}/code/Packages/wsd')

In [3]:
import glob
import time
import json

import pandas as pd
from tqdm import tqdm 
tqdm.pandas()

In [6]:
from importlib import reload

import wsd_test as wsd
import preprocessing_nopos as preprocessing

# Concat all communities

In [7]:
# Open node list
nodelist = pd.read_pickle(project_dir+'/data/3 - Network Generation/final_edgelists/nodelist.pkl')
res = dict((v,k) for k,v in nodelist.items())

In [8]:
def json_to_dict(filename):
  with open(filename) as json_file:
    data = json.load(json_file)
    data['ego'] = res.get(data['ego'])
    target_words: data['ego']
    # print("Ego:", data['ego'])
    # Convert to list of words
    wordlist = data['community'].items()
    commlist = []
    for key, community in wordlist: #converts ids inside the community to words
      new_dict = {}
      new_dict['id'] = key
      new_dict['context_words'] = [res.get(x) for x in community]
      commlist.append(new_dict)
    data['community'] = commlist
    # print("Communities:", data['community'])
    return data
def assign_algorithm(filename):
  return "leiden_mod"

In [12]:
wsi_comms = [] # Stores all the communities in sentence format instead of IDs
for file in os.listdir(project_dir+"/data/4 - Word Sense Induction/communities/leiden_mod_filtered_10/"):
    filename = os.fsdecode(file)
    data = json_to_dict(project_dir+"/data/4 - Word Sense Induction/communities/leiden_mod_filtered_10/"+filename)
    data['algorithm'] = assign_algorithm(filename)
    wsi_comms.append(data)

In [10]:
with open(f'{project_dir}/data/6 - Sense Creation/wsi_comms_lm.json', "w") as outfile:
    json.dump(wsi_comms, outfile)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ASUS\\Downloads\\SOURCE/data/6 - Sense Creation/wsi_comms_lm.json'

# Map Sentences to Word

In [13]:
edgelist_directory = f"{project_dir}/data/6 - Sense Creation/edgelist_wsd"

In [14]:
#for iterative procesing
def save_edgelist(df, month):
    source = df["source"].iloc[0] 
    source_type = df["source_type"].iloc[0]
    year = int(df["year"].iloc[0])

    if not os.path.exists(f"{edgelist_directory}/{source_type}/{source}/{year}/{month}"): 
        os.makedirs(f"{edgelist_directory}/{source_type}/{source}/{year}/{month}") 
    df.dropna().to_csv(f"{edgelist_directory}/{source_type}/{source}/{year}/{month}/{source_type}_{source}_{year}_{month}.csv", index=False)
    print(f'File saved as {source_type}_{source}_{year}_{month}.csv')

In [15]:
#for single-file processing
def save_edgelist_2(df, month, index):
    source = df["source"].iloc[0] 
    source_type = df["source_type"].iloc[0]
    year = int(df["year"].iloc[0])

    if not os.path.exists(f"{edgelist_directory}/{source_type}/{source}/{year}/{month}"): 
        os.makedirs(f"{edgelist_directory}/{source_type}/{source}/{year}/{month}") 
    df.dropna().to_csv(f"{edgelist_directory}/{source_type}/{source}/{year}/{month}/{source_type}_{source}_{year}_{month}_{index}.csv", index=False)
    print(f'File saved as {source_type}_{source}_{year}_{month}.csv')

### Iterative Processing

In [16]:
cohfie_directory = f"{project_dir}/data/2 - COHFIE Creation/COHFIE"
source_type = 'books' #change source here
source = 'google_books' #change source type here
path = f'{cohfie_directory}/{source_type}/{source}'

# dataframes = []  # a list to hold all the individual pandas DataFrames

# directory iterate from year down to months 
years = sorted(os.listdir(path))
for year in years:
    months = sorted(os.listdir(f'{path}/{year}'))

    for month in months:
        dataframes = []  # a list to hold all the individual pandas DataFrames
        file_path = f'{path}/{year}/{month}'
        json_files = sorted(glob.glob(f'{file_path}/*[!_POS].json'))
        pos_files = sorted(glob.glob(f'{file_path}/*_POS.json'))
        print("DATE: ", year, "/", month)
        # loop through the files and read them in with pandas
        for index in range(len(json_files)):
            print(f"Importing {json_files[index].split('/')[-1]}...")
            pos_df = pd.read_json(open(pos_files[index]), orient="records") 
            original_df = pd.read_json(open(json_files[index]), orient="records").drop(['pos_tags'], axis=1) 

            df = pos_df.join(original_df)
            df = df.loc[(((df['lang'] == 'en') & (df['lang_prob'] < 0.8)) | (df['lang'] == 'fil')) & (df['pos_tags'].map(len) > 4)]
            dataframes.append(df)

        try:
            edgelist_df = wsd.create_edgelist(pd.concat(dataframes, ignore_index=True))
            save_edgelist(edgelist_df.explode('word_list'), month)
        except Exception:
            pass
        

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\ASUS\\Downloads\\SOURCE/data/2 - COHFIE Creation/COHFIE/books/google_books'

### Single File Processing

In [ ]:
cohfie_directory = f"{project_dir}/data/2 - COHFIE Creation/COHFIE"
source_type = 'social_media' #change source here
source = 'twitter' #change source type here
year = 2021 #change year here
month = 9 #change month here
file_path = f'{cohfie_directory}/{source_type}/{source}/{year}/{month}'

# get all the csv files in that directory (assuming they have the extension .json)
json_files = sorted(glob.glob(f'{file_path}/*[!_POS].json'))
pos_files = sorted(glob.glob(f'{file_path}/*_POS.json'))
print("DATE: ", year, "/", month)

print(file_path)

dataframes = []  # a list to hold all the individual pandas DataFrames
for index in range(len(json_files)):
    print(index)
    pos_df = pd.read_json(open(f'{file_path}/{source}_{source_type}_{year}_{month}_{index}_POS.json'), orient="records") 
    display(pos_df)
    original_df = pd.read_json(open(f'{file_path}/{source}_{source_type}_{year}_{month}_{index}.json'), orient="records").drop(['pos_tags'], axis=1) 
    display(original_df)
    df = pos_df.join(original_df)
    df = df.loc[(((df['lang'] == 'en') & (df['lang_prob'] < 0.8)) | (df['lang'] == 'fil')) & (df['pos_tags'].map(len) > 4)]
    dataframes = [df]
    edgelist_df = wsd.create_edgelist(pd.concat(dataframes, ignore_index=True))
    id = "_" + str(index)
    save_edgelist(edgelist_df.explode('word_list'), month, id)
    

# Segregate sentences to their ego 

In [ ]:
def save_df(df):
    source = df["source"].iloc[0] 

    df = df.drop_duplicates()
    df.dropna().to_csv(f"{edgelist_directory}/{source}.csv", index=False)
    print(f'File saved as {source_type}.csv')

In [ ]:
#save all sentences in one .csv file for each sources
source = 'social_media' #change
path = f'{edgelist_directory}/{source}'

dataframes = pd.DataFrame()
sources = sorted(os.listdir(path))

for source in sources:
  years = sorted(os.listdir(f'{path}/{source}/'))

  for year in years:
      months = sorted(os.listdir(f'{path}/{source}/{year}'))

      for month in months:
        files = sorted(glob.glob(f'{path}/{source}/{year}/{month}/*.csv'))

        for index in range(len(files)):
            print(f"Importing {files[index].split('/')[-1]}...")
            df = pd.read_csv(open(files[index])) 
            dataframes = dataframes.append(df)

save_df(dataframes)

In [ ]:
#save all sentences in one dataframe
dataframes = pd.DataFrame()

files = sorted(glob.glob(f'{edgelist_directory}/*.csv'))

for index in range(len(files)):
    print(f"Importing {files[index].split('/')[-1]}...")
    df = pd.read_csv(open(files[index])) 
    dataframes = dataframes.append(df)

Importing books.csv...
Importing news_sites.csv...
Importing online_forums.csv...
Importing social_media.csv...
Importing wikipedia.csv...


In [ ]:
if not os.path.exists(f"{project_dir}/data/6 - Sense Creation/wsd_sentences"): 
    os.makedirs(f"{project_dir}/data/6 - Sense Creation/wsd_sentences") 
sentences_directory = f"{project_dir}/data/6 - Sense Creation/wsd_sentences"

for word in ego_list:
  if not os.path.isfile(f"{sentences_directory}/{word}.csv"):
    sentences = dataframes[dataframes['word_list'] == word]
    if not os.path.exists(f"{sentences_directory}"): 
        os.makedirs(f"{sentences_directory}") 
    sentences.to_csv(f"{sentences_directory}/{word}.csv", index=False)



# Load Communities

In [ ]:
#load all the communities
with open(f'{project_dir}/data/6 - Sense Creation/wsi_comms_lm.json') as json_file:
    wsi_comms = json.load(json_file)

# Word Sense Disambiguation

In [ ]:
# Code in this cell is from the package
def jaccard_similarity(comm1, comm2):
    intersection = len(set(comm1).intersection(set(comm2)))
    #union = len(set(comm1)) + len(set(comm2)) - intersection
    union = len(set(comm1).union(set(comm2)))
    return float(intersection) / union if union > 0 else -1
    #return len(comm1) * intersection / union if union > 0 else -1

def get_similar_comm(context_words, comms, target):
    max_index = 0
    max_score = 0
    similarity_scores = []
    for i in range(len(comms)):
      sim = jaccard_similarity(word_tokenize_with_filter(context_words, target), comms[i]['context_words'])
      similarity_scores.append(sim)
      #print("community id: ",i, sim)
      max_score = max(similarity_scores)
      max_index = similarity_scores.index(max_score)
    #print(f"Max is {max_score} of Comm ID {max_index}")
    return max_index, max_score

def word_sense_disambiguation(sentences, comms):
    final_scores = {}
    scores = {}
    for row in sentences.itertuples():
      label, score = get_similar_comm(row.text, comms['community'], comms['ego'])
      scores[row] = label
      final_scores[row] = score
    return scores, final_scores, comms['algorithm'], comms['ego']


def word_tokenize_with_filter(sentence, target):
    words = []
    sentence = sentence.split(' ')
    sent = [wsd.normalize(word.lower().strip()).strip() for word in sentence]
    for index in range(len(sent)):
        word = sent[index]
        if wsd.is_valid(word):
            words.append(word)
    # Manipulate indices to just retain the window size 3 and REMOVE the target word
    return words

In [ ]:
sentences_list = sorted(glob.glob(f'{sentences_directory}/*.csv'))

In [ ]:
def remove_dups(x):
  return list(dict.fromkeys(x))

In [ ]:
if not os.path.exists(f"{project_dir}/data/6 - Sense Creation/sense_inventory"): 
    os.makedirs(f"{project_dir}/data/6 - Sense Creation/sense_inventory") 
directory = f"{project_dir}/data/6 - Sense Creation/sense_inventory"

for index in range(len(sentences_list)):
    print(f"{index} - Importing {sentences_list[index].split('/')[-1]}...")
    sentences = pd.read_csv(open(sentences_list[index])) 
    if len(sentences) != 0:
      comms = list(filter(lambda comm: comm['ego'] == sentences['word_list'][0], wsi_comms))

      temp_comms = pd.DataFrame.from_dict(comms)

      word_results = []

      if len(comms[0]['community']) != 0:
        for comm in comms:
          #res,algo, ego = word_sense_disambiguation(test_db_sentences['sentences'], comm)
          res, score, algo, ego = word_sense_disambiguation(sentences, comm)
          temp = {}
          temp['word'] = ego
          temp['algorithm'] = algo
          temp['results'] = [k for k,v in res.items()]
          temp['comm'] = [v for k,v in res.items()]
          temp['sim_score'] = [v for k,v in score.items()]
          word_results.append(temp)

        df = pd.DataFrame(word_results)

        temp_df = df.explode(['results', 'comm', 'sim_score'], ignore_index=True) 
        temp_df['sim_score'] = temp_df['sim_score'].astype(float) #convert to float


        top10_df = pd.DataFrame()
        for comm in temp_df['comm'].unique():

          #get context details
          context_details = pd.DataFrame()
          for index, row in temp_df[temp_df['comm'] == comm].iterrows():
            temp_contextdict = {"source": row['results'][4], "type": row['results'][3],"year": int(row['results'][5])} #context_details
            context_details = context_details.append(temp_contextdict, ignore_index = True)
          
          temp_contextdf = context_details.groupby(['source', 'type', 'year']).size().reset_index(name='size') #get count
          temp_contextdf['year'] = temp_contextdf['year'].astype(int) #convert to int to avoid decimals

          context_details = {}
          type_values = {}
          year_values = {}
          for (index, row), ii in zip(temp_contextdf.iterrows(), range(len(temp_contextdf.index))):
            if ii == 0:
              year_values.update({row['year']: row['size']})
              type_values.update({row['type']: year_values})
            if ii != 0 and row['source'] != temp_contextdf.loc[ii-1, 'source']:
              type_values = {}
            if ii != 0 and row['type'] != temp_contextdf.loc[ii-1, 'type']:
              year_values = {}
              year_values.update({row['year']: row['size']})
              type_values.update({row['type']: year_values})
              context_details.update({row['source']: type_values})
            else:
              year_values.update({row['year']: row['size']})
          
          #get sentences
          temp_df.drop_duplicates(subset=['results'])
          top_df = temp_df[temp_df['comm'] == comm].nlargest(n=20, columns=['sim_score']) #get top 20 as safety measures in case there are duplicates within top 10

          sense_df = pd.DataFrame()
          sentences = []
          for index, row in top_df.iterrows():
            sentences.append(row['results'][1])

          sentences = remove_dups(sentences) #remove duplicate sentences

          #sense_id
          sense_id = "ns_" + row['word'] + "_" + str(row['comm']) 

          #get community
          a = temp_comms['community'].explode('id') 
          b = pd.DataFrame(a)
          community = b['community'][row['comm']]['context_words']

          sense_df = {
                "sense_id": sense_id,
                "word": row['word'],
                "community": community,
                "example_sentences": sentences[:10],
                "contextual_info": context_details,
            }

          with open(f'{directory}/{sense_id}.json', 'w') as fp:
            json.dump(sense_df, fp)

3998 - Importing sukarno.csv...
3999 - Importing sukat.csv...
4000 - Importing sukdulan.csv...
4001 - Importing suki.csv...
4002 - Importing suklay.csv...
4003 - Importing suko.csv...
4004 - Importing sulat-kamay.csv...
4005 - Importing sulat.csv...
4006 - Importing sulatin.csv...
4007 - Importing suliranin.csv...
4008 - Importing sulla.csv...
4009 - Importing sullivan.csv...
4010 - Importing sully.csv...
4011 - Importing sulok.csv...
4012 - Importing sulong.csv...
4013 - Importing sulyap.csv...
4014 - Importing sumakit.csv...
4015 - Importing sumalpok.csv...
4016 - Importing sumalungat.csv...
4017 - Importing sumang-ayon.csv...
4018 - Importing sumara.csv...
4019 - Importing sumbrero.csv...
4020 - Importing sumigaw.csv...
4021 - Importing sumipsip.csv...
4022 - Importing sumisidhi.csv...
4023 - Importing sumner.csv...
4024 - Importing sumo.csv...
4025 - Importing sumpa.csv...
4026 - Importing sumpong.csv...
4027 - Importing sumulak.csv...
4028 - Importing sumulat.csv...
4029 - Importi